# OOF với class weight ở tầng 1

**Mục đích:** sinh xác suất OOF cho các base model khi **có class weight**, để cải thiện macro-F1 của tầng 1 (không chỉ tầng 2).

**Hai chế độ weight:**

| Mode | Trọng số | Ghi chú |
|---|---|---|
| `balanced` | `n/(k·count)` | Class 1 ≈ 17×, class 2 ≈ 3.2× — mạnh tay |
| `sqrt` | `sqrt(balanced)` | Class 1 ≈ 4.1×, class 2 ≈ 1.8× — nhẹ hơn, an toàn hơn |

**Vì sao cần cả dev/test:** stacking train trên OOF, nhưng dự đoán dev/test phải đến từ **cùng chế độ huấn luyện**. Notebook này lưu thêm `{model}_dev.npy` và `{model}_test.npy` (trung bình qua 5 fold) vào cùng thư mục.

**Đầu ra:**
- `artifacts/probs/oof_balanced/{model}_{oof,dev,test}.npy`
- `artifacts/probs/oof_sqrt/{model}_{oof,dev,test}.npy`

**Thời gian:** mỗi mode ~80 phút (LightGBM chiếm ~45 phút). Có thể chạy riêng từng mode và từng model; cell tự bỏ qua model đã có file.

> Không ghi đè bộ OOF gốc `artifacts/probs/oof/`.

In [18]:
# ============================================================
# 1. IMPORT & CẤU HÌNH
# ============================================================
import time
import warnings
from importlib.util import find_spec
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.naive_bayes import ComplementNB
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")
pd.set_option("display.width", 180)
SEED = 42
N_SPLITS = 5
MODEL_NAMES = ["logreg", "linsvc", "sgd", "complement_nb", "lightgbm"]
MODELS_TO_RUN = MODEL_NAMES.copy()   # sửa danh sách này nếu chỉ muốn chạy vài model
SKIP_IF_EXISTS = True
HAVE_LGBM = find_spec("lightgbm") is not None
print("lightgbm khả dụng:", HAVE_LGBM)


def find_project_root(start=None, marker="Data/ds102_train.csv"):
    p = Path(start).resolve() if start is not None else Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / marker).exists():
            return cand
    raise FileNotFoundError(f"Không tìm thấy thư mục gốc chứa '{marker}'")


PROJECT_ROOT = find_project_root()
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
PROBS_ROOT = ARTIFACT_DIR / "probs"

print("PROJECT_ROOT:", PROJECT_ROOT)

lightgbm khả dụng: True
PROJECT_ROOT: D:\DS102_NguyenKhacHuy_24520679\Final project


In [19]:
# ============================================================
# 2. NẠP DỮ LIỆU
# ============================================================
X_train = sparse.load_npz(ARTIFACT_DIR / "sparse" / "X_train.npz").tocsr()
X_dev = sparse.load_npz(ARTIFACT_DIR / "sparse" / "X_dev.npz").tocsr()
X_test = sparse.load_npz(ARTIFACT_DIR / "sparse" / "X_test.npz").tocsr()
y_train = np.load(ARTIFACT_DIR / "labels" / "y_train.npy")
pre = joblib.load(ARTIFACT_DIR / "models" / "preprocessors.joblib")

N_DENSE = len(pre["dense_feature_names"])
N_TRAIN = X_train.shape[0]
N_DEV = X_dev.shape[0]
N_TEST = X_test.shape[0]

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
SPLITS = list(skf.split(X_train, y_train))

print("X_train:", X_train.shape, "| X_dev:", X_dev.shape, "| X_test:", X_test.shape)
print("Số fold:", N_SPLITS)

X_train: (14306, 98713) | X_dev: (1590, 98713) | X_test: (3974, 98713)
Số fold: 5


In [20]:
# ============================================================
# 3. CLASS WEIGHT & MODEL FACTORY
# ============================================================
def class_weight_dict(mode, y):
    counts = np.bincount(y, minlength=4).astype(float)
    balanced = len(y) / (4.0 * np.maximum(counts, 1.0))
    if mode == "balanced":
        vals = balanced
    elif mode == "sqrt":
        vals = np.sqrt(balanced)
    else:
        raise ValueError(f"mode không hợp lệ: {mode}")
    return {i: float(vals[i]) for i in range(4)}


def make_model(name, cw):
    if name == "logreg":
        return LogisticRegression(C=1.0, class_weight=cw, max_iter=1000,
                                  solver="lbfgs", random_state=SEED)
    if name == "linsvc":
        return CalibratedClassifierCV(
            estimator=LinearSVC(C=0.5, class_weight=cw, max_iter=5000, random_state=SEED),
            cv=3, method="sigmoid",
        )
    if name == "sgd":
        return SGDClassifier(loss="modified_huber", alpha=1e-5, max_iter=3000,
                             tol=1e-4, class_weight=cw, random_state=SEED)
    if name == "lightgbm":
        from lightgbm import LGBMClassifier
        return LGBMClassifier(
            n_estimators=400, learning_rate=0.05, num_leaves=63,
            subsample=0.8, colsample_bytree=0.8, class_weight=cw,
            random_state=SEED, n_jobs=-1, verbose=-1,
        )
    if name == "complement_nb":
        return ComplementNB(alpha=1.0)  # dùng sample_weight khi fit
    raise ValueError(name)


def assemble(tr_idx, va_idx):
    scaler = StandardScaler(with_mean=False).fit(X_train[tr_idx][:, :N_DENSE])
    A_tr = sparse.hstack(
        [scaler.transform(X_train[tr_idx][:, :N_DENSE]), X_train[tr_idx][:, N_DENSE:]],
        format="csr",
    ).astype(np.float32)
    A_va = sparse.hstack(
        [scaler.transform(X_train[va_idx][:, :N_DENSE]), X_train[va_idx][:, N_DENSE:]],
        format="csr",
    ).astype(np.float32)
    A_dv = sparse.hstack(
        [scaler.transform(X_dev[:, :N_DENSE]), X_dev[:, N_DENSE:]],
        format="csr",
    ).astype(np.float32)
    A_te = sparse.hstack(
        [scaler.transform(X_test[:, :N_DENSE]), X_test[:, N_DENSE:]],
        format="csr",
    ).astype(np.float32)
    A_va_tfidf = X_train[va_idx][:, N_DENSE:]
    A_dv_tfidf = X_dev[:, N_DENSE:]
    A_te_tfidf = X_test[:, N_DENSE:]
    return A_tr, A_va, A_dv, A_te, A_va_tfidf, A_dv_tfidf, A_te_tfidf


print("Đã định nghĩa factory.")

Đã định nghĩa factory.


## 4. Hàm chạy OOF cho một model + một mode

Mỗi fold:
- fit scaler + model trên fold train,
- dự đoán fold validation → điền vào OOF,
- dự đoán dev và test → cộng dồn, cuối cùng chia cho 5.

Nếu đã có đủ 3 file và `SKIP_IF_EXISTS=True`, bỏ qua.

In [21]:
# ============================================================
# 4. HÀM CHẠY OOF
# ============================================================
RESULTS = {}


def run_oof(name, mode):
    if name == "lightgbm" and not HAVE_LGBM:
        print(f"[{mode}] lightgbm chưa cài — bỏ qua")
        return None
    out_dir = PROBS_ROOT / f"oof_{mode}"
    out_dir.mkdir(parents=True, exist_ok=True)
    f_oof = out_dir / f"{name}_oof.npy"
    f_dev = out_dir / f"{name}_dev.npy"
    f_test = out_dir / f"{name}_test.npy"

    if SKIP_IF_EXISTS and f_oof.exists() and f_dev.exists() and f_test.exists():
        oof = np.load(f_oof)
        mf1 = f1_score(y_train, oof.argmax(axis=1), average="macro")
        acc = accuracy_score(y_train, oof.argmax(axis=1))
        print(f"[{mode}] {name:14s} đã có file — bỏ qua | OOF mf1={mf1:.4f} acc={acc:.4f}")
        RESULTS[f"{mode}/{name}"] = {"macro_f1": round(float(mf1), 4), "accuracy": round(float(acc), 4)}
        return oof

    oof = np.zeros((N_TRAIN, 4), dtype=np.float32)
    dev_sum = np.zeros((N_DEV, 4), dtype=np.float32)
    test_sum = np.zeros((N_TEST, 4), dtype=np.float32)
    t_start = time.time()

    for k, (tr_idx, va_idx) in enumerate(SPLITS):
        t0 = time.time()
        cw = class_weight_dict(mode, y_train[tr_idx])
        model = make_model(name, cw)
        A_tr, A_va, A_dv, A_te, A_va_t, A_dv_t, A_te_t = assemble(tr_idx, va_idx)

        if name == "complement_nb":
            sw = np.array([cw[int(v)] for v in y_train[tr_idx]], dtype=np.float64)
            model.fit(X_train[tr_idx][:, N_DENSE:], y_train[tr_idx], sample_weight=sw)
            p_va = model.predict_proba(A_va_t)
            p_dv = model.predict_proba(A_dv_t)
            p_te = model.predict_proba(A_te_t)
        else:
            model.fit(A_tr, y_train[tr_idx])
            p_va = model.predict_proba(A_va)
            p_dv = model.predict_proba(A_dv)
            p_te = model.predict_proba(A_te)

        oof[va_idx] = p_va
        dev_sum += p_dv / N_SPLITS
        test_sum += p_te / N_SPLITS

        fold_mf1 = f1_score(y_train[va_idx], p_va.argmax(axis=1), average="macro")
        print(f"  [{mode}] {name:14s} fold {k}: mf1={fold_mf1:.4f} ({time.time() - t0:.0f}s)")

    oof_mf1 = f1_score(y_train, oof.argmax(axis=1), average="macro")
    oof_acc = accuracy_score(y_train, oof.argmax(axis=1))
    print(f"[{mode}] {name:14s} OOF mf1={oof_mf1:.4f} acc={oof_acc:.4f} "
          f"| tổng {time.time() - t_start:.0f}s")

    np.save(f_oof, oof)
    np.save(f_dev, dev_sum)
    np.save(f_test, test_sum)
    RESULTS[f"{mode}/{name}"] = {"macro_f1": round(float(oof_mf1), 4), "accuracy": round(float(oof_acc), 4)}
    return oof

## 5. Chạy mode `balanced`

Class 1 weight ≈ 17×. Mạnh tay — theo dõi precision của class 1.

Cell tự bỏ qua model đã có file, nên có thể chạy lại an toàn.

In [22]:
assert "run_oof" in globals(), (
    "Chưa chạy các cell trước. Hãy chạy lần lượt cell 1 (import), 2 (dữ liệu), "
    "3 (factory), 4 (hàm run_oof) rồi mới chạy cell này."
)
print("Chạy mode balanced cho:", MODELS_TO_RUN)
for name in MODELS_TO_RUN:
    run_oof(name, "balanced")

Chạy mode balanced cho: ['logreg', 'linsvc', 'sgd', 'complement_nb', 'lightgbm']
  [balanced] logreg         fold 0: mf1=0.7452 (168s)
  [balanced] logreg         fold 1: mf1=0.7132 (92s)
  [balanced] logreg         fold 2: mf1=0.7175 (89s)
  [balanced] logreg         fold 3: mf1=0.7430 (93s)
  [balanced] logreg         fold 4: mf1=0.6980 (90s)
[balanced] logreg         OOF mf1=0.7240 acc=0.8645 | tổng 531s
  [balanced] linsvc         fold 0: mf1=0.7363 (256s)
  [balanced] linsvc         fold 1: mf1=0.7092 (238s)
  [balanced] linsvc         fold 2: mf1=0.7027 (251s)
  [balanced] linsvc         fold 3: mf1=0.7092 (241s)
  [balanced] linsvc         fold 4: mf1=0.6883 (238s)
[balanced] linsvc         OOF mf1=0.7098 acc=0.8906 | tổng 1223s
  [balanced] sgd            fold 0: mf1=0.7198 (5s)
  [balanced] sgd            fold 1: mf1=0.6930 (6s)
  [balanced] sgd            fold 2: mf1=0.6434 (7s)
  [balanced] sgd            fold 3: mf1=0.4489 (7s)
  [balanced] sgd            fold 4: mf1=0.6878

## 6. Chạy mode `sqrt`

Weight nhẹ hơn (class 1 ≈ 4.1×, class 2 ≈ 1.8×), thường an toàn hơn với class 1 chỉ có 211 mẫu.

In [23]:
assert "run_oof" in globals(), (
    "Chưa chạy các cell trước. Hãy chạy lần lượt cell 1 (import), 2 (dữ liệu), "
    "3 (factory), 4 (hàm run_oof) rồi mới chạy cell này."
)
print("Chạy mode sqrt cho:", MODELS_TO_RUN)
for name in MODELS_TO_RUN:
    run_oof(name, "sqrt")

Chạy mode sqrt cho: ['logreg', 'linsvc', 'sgd', 'complement_nb', 'lightgbm']
  [sqrt] logreg         fold 0: mf1=0.7546 (101s)
  [sqrt] logreg         fold 1: mf1=0.7345 (100s)
  [sqrt] logreg         fold 2: mf1=0.7509 (95s)
  [sqrt] logreg         fold 3: mf1=0.7467 (98s)
  [sqrt] logreg         fold 4: mf1=0.7252 (97s)
[sqrt] logreg         OOF mf1=0.7433 acc=0.8868 | tổng 492s
  [sqrt] linsvc         fold 0: mf1=0.7369 (257s)
  [sqrt] linsvc         fold 1: mf1=0.7115 (317s)
  [sqrt] linsvc         fold 2: mf1=0.7019 (299s)
  [sqrt] linsvc         fold 3: mf1=0.7182 (251s)
  [sqrt] linsvc         fold 4: mf1=0.6950 (240s)
[sqrt] linsvc         OOF mf1=0.7132 acc=0.8913 | tổng 1363s
  [sqrt] sgd            fold 0: mf1=0.6846 (6s)
  [sqrt] sgd            fold 1: mf1=0.5872 (6s)
  [sqrt] sgd            fold 2: mf1=0.6164 (7s)
  [sqrt] sgd            fold 3: mf1=0.5563 (7s)
  [sqrt] sgd            fold 4: mf1=0.6206 (7s)
[sqrt] sgd            OOF mf1=0.6177 acc=0.8612 | tổng 33s
  [sqr

## 7. Tổng hợp OOF

So sánh macro-F1 OOF giữa các mode và các model. Bảng này chỉ để tham khảo — quyết định cuối cùng nằm ở notebook stacking (dev macro-F1).

In [24]:
if RESULTS:
    summary = pd.DataFrame(RESULTS).T.sort_values("macro_f1", ascending=False)
    print(summary.to_string())
else:
    print("Chưa chạy model nào.")

for mode in ["balanced", "sqrt"]:
    d = PROBS_ROOT / f"oof_{mode}"
    if d.exists():
        print(f"\n[{d}]")
        for p in sorted(d.glob("*.npy")):
            print(f"  {p.name:28s} {p.stat().st_size / 1e6:.2f} MB")

                        macro_f1  accuracy
sqrt/logreg               0.7433    0.8868
balanced/lightgbm         0.7335    0.8987
balanced/logreg           0.7240    0.8645
sqrt/lightgbm             0.7222    0.8972
sqrt/linsvc               0.7132    0.8913
balanced/linsvc           0.7098    0.8906
balanced/sgd              0.6219    0.8550
sqrt/sgd                  0.6177    0.8612
sqrt/complement_nb        0.6051    0.8334
balanced/complement_nb    0.4867    0.6317

[D:\DS102_NguyenKhacHuy_24520679\Final project\artifacts\probs\oof_balanced]
  complement_nb_dev.npy        0.03 MB
  complement_nb_oof.npy        0.23 MB
  complement_nb_test.npy       0.06 MB
  lightgbm_dev.npy             0.03 MB
  lightgbm_oof.npy             0.23 MB
  lightgbm_test.npy            0.06 MB
  linsvc_dev.npy               0.03 MB
  linsvc_oof.npy               0.23 MB
  linsvc_test.npy              0.06 MB
  logreg_dev.npy               0.03 MB
  logreg_oof.npy               0.23 MB
  logreg_test.npy   

## 8. Ghi chú

- Không ghi đè `artifacts/probs/oof/` (bộ gốc).
- `{model}_dev.npy` và `{model}_test.npy` ở đây là **trung bình 5 fold** — nhất quán với OOF, khác với `artifacts/probs/*_dev.npy` (model fit toàn train).
- Nếu một mode cho class 1 precision quá thấp, đó là dấu hiệu weight quá mạnh; dùng mode `sqrt` hoặc giảm weight.
- Stacking sẽ tự phát hiện các thư mục `oof_balanced`, `oof_sqrt` và so sánh.